# 1. Post-Reindexing User Vector Migration

The ReIndexing Solution migrates face vectors to a new collection using `IndexFaces`, but does not recreate [User Vectors](https://aws.amazon.com/about-aws/whats-new/2023/06/amazon-rekognition-face-search-accuracy-user-vectors/). This notebook reads the DynamoDB results table produced by the reindexing process and recreates users and face-to-user associations in the target collection.

**Notebook Steps:**
1. Import libraries and define variables
2. Read reindexing results from DynamoDB
3. Create users in the target collection
4. Associate faces to users
5. Verify the target collection

**Prerequisites:**
- The ReIndexing Solution has completed successfully
- The DynamoDB results table contains the old-to-new FaceId mappings with UserId

**NOTE: You can run this notebook in SageMaker Studio, JupyterLab, or on your local machine**

### 1. Import libraries and define variables

In [ ]:
import boto3
from collections import defaultdict

region = 'us-west-2'
session = boto3.Session(region_name=region)
dynamodb_client = session.client('dynamodb')
rek_client = session.client('rekognition')

In [ ]:
# DynamoDB table created by the ReIndexing Solution CloudFormation stack
results_table_name = ""  # e.g. "RIS-<stack-name>-ReIndexResults"

# Target collection where faces were reindexed
target_collection_id = ""  # e.g. "my-new-collection"

### 2. Read reindexing results from DynamoDB

Scan the results table and group the new FaceIds by UserId.

In [ ]:
user_faces = defaultdict(list)
paginator = dynamodb_client.get_paginator('scan')

for page in paginator.paginate(TableName=results_table_name):
    for item in page['Items']:
        user_id = item.get('UserID', {}).get('S')
        face_id = item.get('FaceId', {}).get('S')
        if user_id and face_id:
            user_faces[user_id].append(face_id)

print(f"Found {len(user_faces)} users:")
for user_id, faces in user_faces.items():
    print(f"  {user_id}: {len(faces)} faces - {faces}")

### 3. Create users in the target collection

In [ ]:
for user_id in user_faces.keys():
    try:
        rek_client.create_user(CollectionId=target_collection_id, UserId=user_id)
        print(f"Created user: {user_id}")
    except rek_client.exceptions.ResourceAlreadyExistsException:
        print(f"User already exists: {user_id}")

### 4. Associate faces to users

In [ ]:
for user_id, face_ids in user_faces.items():
    try:
        response = rek_client.associate_faces(
            CollectionId=target_collection_id,
            UserId=user_id,
            FaceIds=face_ids
        )
        associated = len(response.get('AssociatedFaces', []))
        print(f"Associated {associated}/{len(face_ids)} faces to user: {user_id}")
    except Exception as e:
        print(f"Error associating faces for {user_id}: {e}")

### 5. Verify the target collection

In [ ]:
desc = rek_client.describe_collection(CollectionId=target_collection_id)
print(f"Collection: {target_collection_id}")
print(f"  FaceCount: {desc['FaceCount']}")
print(f"  UserCount: {desc['UserCount']}")
print(f"  FaceModelVersion: {desc['FaceModelVersion']}")